<div style="text-align:center">
<img src="img/nlopt3.gif" width="600">
</div>

# <b> Introduction</b>

In this notebook, I will check " what A.I is made of" . I study one or 2 cases a day , trying to do some comparisons, studying maths models, and instances , how A.I deals with non linear combinatorial problems explanations of things. I have books, but I won't use them in this particular notebook.

# <b> Summary </b>

01. <b>Gemini A.I : Quadratic production scheduling optimization involving risk </b>
    - Abstract
    - Instance
    - Math model
    - Docplex solution

02. <b>Gemini A.I : Quadratic Phone calling center optimization </b>
    - Abstract
    - Instance
    - Math model
    - Docplex solution


03. <b>Gemini A.I : Quadratic diet optimization </b>
    - Abstract
    - Instance
    - Math model
    - Docplex solution

04. <b>Gemini A.I : Quadratic Gasoline Blending with Octane Synergy </b>
    - Abstract
    - Instance
    - Math model
    - Docplex solution

# <b>01.  Gemini A.I : Quadratic production scheduling optimization involving risk </b> 

<div style="text-align:center">
<img src="img/toyfactory.jpg" width="400">
</div>

<div style="text-align:center">
<img src="img/gemini.png" width="200"/>
</div>

## <b>Abstract</b>
<i>Can you provide a quadratic production optimisation involving risk , using docplex and describe the math model ? </i>

<div style = "background:green;color:white">
Gemini said :
In production planning, optimization isn't just about maximizing output; it's about balancing expected profit against the volatility (risk) of that profit. This is a classic application of Quadratic Programming (QP).
</div>

<b>Estelle note : This is a Markowitz model  applied to a production mix, I recognize the habitual covariance matrix, and the risk rate aversion</b>


## <b>Instance</b>
We want to decide the production levels for a set of products to maximize return while penalizing high variance in profit (often caused by fluctuating market prices or resource costs).



## <b> Math model </b>

<div style="text-align:center">
<img src="img/Screenshot 2026-03-05 at 20-38-31 Google Gemini.png" width=""/>
</div>
<div style="text-align:center">
<img src="img/Screenshot 2026-03-05 at 20-39-18 Google Gemini.png" width=""/>
</div>




## <b> Docplex solution </b>

<div style="text-align:center">
<img src="img/docplex-banner.webp" width="200"/>
</div>

In [4]:
from docplex.mp.model import Model
import numpy as np

# 1. Data Setup
products = ['Product_A', 'Product_B', 'Product_C']
avg_profits = {'Product_A': 50, 'Product_B': 60, 'Product_C': 40}

# Covariance Matrix (representing risk/volatility)
# High values mean the products' profits are volatile or fluctuate together
cov_matrix = [
    [10, 2, 1],  # Product A variance and covariances
    [2, 15, 3],  # Product B
    [1, 3, 8]    # Product C
]

risk_aversion = 0.5  # This is our Lambda (λ)
resource_usage = {'Product_A': 2, 'Product_B': 3, 'Product_C': 1}
max_resource = 100

# 2. Initialize Model
mdl = Model(name='Production_Risk_Optimization')

# 3. Decision Variables
x = mdl.continuous_var_dict(products, lb=0, name='prod')

# 4. Objective Function
# Expected Profit (Linear part)
expected_profit = mdl.sum(avg_profits[p] * x[p] for p in products)

# Risk/Variance (Quadratic part)
# Calculate: sum(cov[i][j] * x[i] * x[j])
risk_penalty = mdl.sum(cov_matrix[i][j] * x[products[i]] * x[products[j]] 
                       for i in range(len(products)) 
                       for j in range(len(products)))

mdl.maximize(expected_profit - risk_aversion * risk_penalty)

# 5. Constraints
mdl.add_constraint(mdl.sum(resource_usage[p] * x[p] for p in products) <= max_resource, "Capacity")

# 6. Solve
solution = mdl.solve()

# 7. Results
if solution:
    print("\n--- Optimal Production Plan ---")
    for p in products:
        print(f"{p}: {solution[x[p]]:.2f} units")
    print(f"Total Objective Value: {solution.objective_value:.2f}")
else:
    print("No solution found.")


--- Optimal Production Plan ---
Product_A: 4.10 units
Product_B: 2.76 units
Product_C: 3.45 units
Total Objective Value: 254.47


- Key Insights for Your Model

    The "Efficient Frontier": If you vary λ from 0 to a high number, you can plot the relationship between "Risk" and "Return." This helps stakeholders choose a production mix that fits their specific comfort level.

    Diversification: Because of the quadratic risk term, the solver will naturally try to "diversify" production. If Product B is very profitable but extremely volatile (high variance), the model will likely shift some production to Product C to stabilize the overall portfolio.

    Why Docplex? Docplex is particularly good at this because it handles Second-Order Cone Programming (SOCP) and QP problems very efficiently compared to basic linear solvers.

Would you like to see how to incorporate "Conditional Value at Risk" (CVaR) instead? That's a slightly more modern way to handle risk if you have historical data rather than just a covariance matrix.